# [Experiment] Parameter-Matched KAN-TabNet | Step LR | Forest Cover

### Overview
This notebook introduces the Kolmogorov-Arnold Network (KAN) architecture into the TabNet framework. The network is specifically configured to match the total trainable parameter count of the original vanilla baseline.

### Methodological Context: Strict Isolation
To precisely isolate the effects of the KAN architecture, we maintain the exact same StepLR optimization environment used in our vanilla control—a schedule chosen to strictly adhere to the original TabNet paper's experimental design. By holding the optimization thermodynamics constant and parameter-matching the models, we guarantee that any performance delta is driven purely by the mathematical flexibility of the B-splines, rather than a brute-force increase in structural capacity or a shifted learning rate.

### The Hypothesis
We hypothesize that substituting standard linear transformations with learnable activation functions on the network edges will fundamentally alter how the model routes and represents features. This notebook serves to evaluate whether this architectural shift improves predictive accuracy or alters convergence behavior when operating under the exact same foundational constraints defined by the original TabNet authors.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier

dataset = fetch_covtype()

X = dataset.data
# CovType is 1-indexed (1 to 7); PyTorch expects 0-indexed labels
y = dataset.target - 1

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (464809, 54)
Valid shape: (58101, 54)
Test shape:  (58102, 54)


### Model Training

In [5]:
MAX_EPOCHS = 1000

In [6]:
# Hyperparameters from original paper.
# Notes:
# - momentum is 1 - 0.7 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 18 epochs approximates the paper's 500 iterations
#   (based on a batch size of 16384 and ~464k training samples).
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-4,
    'momentum': 0.3,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=18, gamma=0.95),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=20, # adjusted so parameter count is matched with vanilla TabNet
    n_a=20, # adjusted so parameter count is matched with vanilla TabNet
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.002), # reduced as 0.02 lr is too aggressive
    use_kan=True,
    kan_grid_size=5,
    kan_spline_order=3,
    seed=seed,
    verbose=50
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=100,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.43491 | valid_accuracy: 0.36023 |  0:00:09s
epoch 50 | loss: 0.18522 | valid_accuracy: 0.92874 |  0:07:07s
epoch 100| loss: 0.13957 | valid_accuracy: 0.94807 |  0:14:04s
epoch 150| loss: 0.11074 | valid_accuracy: 0.95716 |  0:21:01s
epoch 200| loss: 0.10191 | valid_accuracy: 0.95742 |  0:27:57s
epoch 250| loss: 0.10844 | valid_accuracy: 0.95778 |  0:34:55s
epoch 300| loss: 0.08412 | valid_accuracy: 0.96301 |  0:41:51s
epoch 350| loss: 0.07786 | valid_accuracy: 0.96632 |  0:48:48s
epoch 400| loss: 0.06972 | valid_accuracy: 0.96661 |  0:55:45s
epoch 450| loss: 0.06874 | valid_accuracy: 0.96659 |  1:02:42s
epoch 500| loss: 0.06277 | valid_accuracy: 0.96776 |  1:09:40s
epoch 550| loss: 0.06073 | valid_accuracy: 0.96788 |  1:16:37s
epoch 600| loss: 0.05929 | valid_accuracy: 0.96792 |  1:23:34s
epoch 650| loss: 0.07176 | valid_accuracy: 0.96766 |  1:30:32s
epoch 700| loss: 0.05478 | valid_accuracy: 0.96921 |  1:37:30s
epoch 750| loss: 0.05293 | valid_accuracy: 0.97105 |  1

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 469,336


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.97195


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/forest_cover/tables'
MODELS_DIR = f'{BASE_DIR}/results/forest_cover/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Forest Cover",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/02_kan_param_matched_step_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/02_kan_param_matched_step_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/forest_cover/tables/02_kan_param_matched_step_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/forest_cover/models/02_kan_param_matched_step_lr_469k.zip
